In [2]:
import numpy as np
import geopandas as gpd
import mikeio
from shapely.geometry import Polygon

# ============================================================
# INPUT PATHS
# ============================================================
bay_shp_zip = r"D:\Phd Research\GIS\Shape\Major_Bays_polygon.zip"
dfsu_file = r"F:\storm_condition_hurricane_harvey_stat.dfsu"

# ============================================================
# SETTINGS
# ============================================================
bay_field = "bay"
target_bays = ["Matagorda", "San Antonio", "Aransas", "Corpus Christi", "Baffin"]

# ============================================================
# READ BAY POLYGON
# ============================================================
bays_gdf = gpd.read_file(f"zip://{bay_shp_zip}")
bays_gdf = bays_gdf[bays_gdf[bay_field].isin(target_bays)]
bays_gdf = bays_gdf.dissolve(by=bay_field).reset_index()

# ============================================================
# READ DFSU
# ============================================================
dfs = mikeio.open(dfsu_file)

item_names = [it.name for it in dfs.items]

# ---- Explicit item selection based on your screenshot ----
idx_max = item_names.index("Statistical maximum : Total water depth")
idx_mean = item_names.index("Statistical mean : Total water depth")
idx_min = item_names.index("Statistical minimum : Total water depth")

ds = dfs.read(items=[idx_max, idx_mean, idx_min])

val_max = ds[0].values[0]
val_mean = ds[1].values[0]
val_min = ds[2].values[0]

# ============================================================
# BUILD ELEMENT POLYGONS
# ============================================================
geom = dfs.geometry
node_xy = geom.node_coordinates[:, :2]
elem_table = geom.element_table

polygons = []
for nodes in elem_table:
    coords = node_xy[np.asarray(nodes)]
    polygons.append(Polygon(coords))

elem_gdf = gpd.GeoDataFrame({
    "max": val_max,
    "mean": val_mean,
    "min": val_min
}, geometry=polygons, crs=geom.projection_string)

# Reproject bays if needed
if bays_gdf.crs != elem_gdf.crs:
    bays_gdf = bays_gdf.to_crs(elem_gdf.crs)

# ============================================================
# COMPUTE AREA-WEIGHTED AVERAGE
# ============================================================
print("\n===== BAY-WISE STATISTICS (AREA-WEIGHTED) =====\n")

for _, row in bays_gdf.iterrows():
    bay_name = row[bay_field]
    bay_geom = row.geometry

    # Intersect elements
    subset = elem_gdf[elem_gdf.intersects(bay_geom)].copy()

    if subset.empty:
        print(f"{bay_name}: No data")
        continue

    inter_geom = subset.geometry.intersection(bay_geom)
    area = inter_geom.area.values

    mask = area > 0
    subset = subset.loc[mask]
    area = area[mask]

    w = area
    wsum = w.sum()

    avg_max = np.sum(subset["max"].values * w) / wsum
    avg_mean = np.sum(subset["mean"].values * w) / wsum
    avg_min = np.sum(subset["min"].values * w) / wsum

    print(f"{bay_name}")
    print(f"  Avg Statistical MAX  : {avg_max:.3f} m")
    print(f"  Avg Statistical MEAN : {avg_mean:.3f} m")
    print(f"  Avg Statistical MIN  : {avg_min:.3f} m\n")


===== BAY-WISE STATISTICS (AREA-WEIGHTED) =====

Aransas
  Avg Statistical MAX  : 4.119 m
  Avg Statistical MEAN : 3.117 m
  Avg Statistical MIN  : 1.754 m

Baffin
  Avg Statistical MAX  : 3.333 m
  Avg Statistical MEAN : 2.566 m
  Avg Statistical MIN  : 1.380 m

Corpus Christi
  Avg Statistical MAX  : 5.940 m
  Avg Statistical MEAN : 4.909 m
  Avg Statistical MIN  : 3.661 m

Matagorda
  Avg Statistical MAX  : 4.834 m
  Avg Statistical MEAN : 4.011 m
  Avg Statistical MIN  : 2.361 m

San Antonio
  Avg Statistical MAX  : 4.156 m
  Avg Statistical MEAN : 2.828 m
  Avg Statistical MIN  : 1.309 m



In [16]:
import numpy as np
import geopandas as gpd
import mikeio
from shapely.geometry import Polygon
from shapely.validation import make_valid

# ============================================================
# INPUT PATHS
# ============================================================
bay_shp_zip = r"D:\Phd Research\GIS\Shape\Major_Bays_polygon.zip"
dfsu_file = r"F:\storm_condition_hurricane_harvey_Current_speed_stat.dfsu"

# ============================================================
# SETTINGS
# ============================================================
bay_field = "bay"
target_bays = ["Matagorda", "San Antonio", "Aransas", "Corpus Christi", "Baffin"]

# IMPORTANT:
# Set the correct CRS of your DFSU mesh here.
# Use EPSG:32614 if your mesh is UTM Zone 14N.
mesh_epsg = 32614

# small tolerance buffer for intersection robustness (meters)
buffer_tol = 0.0

# ============================================================
# HELPER FUNCTIONS
# ============================================================
def safe_make_polygon(coords):
    """Create valid polygon; return None if impossible."""
    try:
        poly = Polygon(coords)
        if not poly.is_valid:
            poly = make_valid(poly)
        if poly.is_empty:
            return None
        return poly
    except Exception:
        return None

def area_weighted_mean(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not np.any(mask):
        return np.nan
    w = weights[mask]
    v = values[mask]
    wsum = w.sum()
    if wsum <= 0:
        return np.nan
    return np.sum(v * w) / wsum

# ============================================================
# READ BAY POLYGONS
# ============================================================
print("\nReading bay polygons...")
bays_gdf = gpd.read_file(f"zip://{bay_shp_zip}")

if bay_field not in bays_gdf.columns:
    raise ValueError(f"Field '{bay_field}' not found in shapefile. Available fields: {list(bays_gdf.columns)}")

bays_gdf = bays_gdf[bays_gdf[bay_field].isin(target_bays)].copy()
if bays_gdf.empty:
    raise ValueError("No target bays found in shapefile.")

# dissolve by bay name
bays_gdf = bays_gdf.dissolve(by=bay_field).reset_index()

# fix invalid geometries
bays_gdf["geometry"] = bays_gdf["geometry"].apply(lambda g: make_valid(g) if g is not None and not g.is_valid else g)
bays_gdf = bays_gdf[~bays_gdf.geometry.is_empty & bays_gdf.geometry.notnull()].copy()

print("Bay CRS before reprojection:", bays_gdf.crs)

# ============================================================
# READ DFSU
# ============================================================
print("\nReading DFSU...")
dfs = mikeio.open(dfsu_file)
geom = dfs.geometry

item_names = [it.name for it in dfs.items]
print("\nAvailable DFSU items:")
for i, name in enumerate(item_names):
    print(f"  {i}: {name}")

# ---- Find current speed statistical items robustly ----
def find_item_index(possible_names):
    for name in possible_names:
        if name in item_names:
            return item_names.index(name)
    return None

idx_max = find_item_index([
    "Statistical maximum : Current speed",
    "Statistical maximum: Current speed",
    "Maximum current speed"
])

idx_mean = find_item_index([
    "Statistical mean : Current speed",
    "Statistical mean: Current speed",
    "Mean current speed"
])

idx_min = find_item_index([
    "Statistical minimum : Current speed",
    "Statistical minimum: Current speed",
    "Minimum current speed"
])

if idx_max is None or idx_mean is None or idx_min is None:
    raise ValueError(
        "Could not find one or more required current speed items.\n"
        f"Found max index={idx_max}, mean index={idx_mean}, min index={idx_min}"
    )

ds = dfs.read(items=[idx_max, idx_mean, idx_min])

# ------------------------------------------------------------
# Take absolute value first
# ------------------------------------------------------------
val_max = np.abs(np.asarray(ds[0].values[0], dtype=float))
val_mean = np.abs(np.asarray(ds[1].values[0], dtype=float))
val_min = np.abs(np.asarray(ds[2].values[0], dtype=float))

print("\nNaN counts in DFSU values:")
print("  abs(max) :", np.isnan(val_max).sum())
print("  abs(mean):", np.isnan(val_mean).sum())
print("  abs(min) :", np.isnan(val_min).sum())

# ============================================================
# BUILD ELEMENT POLYGONS
# ============================================================
print("\nBuilding DFSU element polygons...")
node_xy = geom.node_coordinates[:, :2]
elem_table = geom.element_table

polygons = []
valid_idx = []

for i, nodes in enumerate(elem_table):
    coords = node_xy[np.asarray(nodes)]
    poly = safe_make_polygon(coords)
    if poly is not None:
        polygons.append(poly)
        valid_idx.append(i)

if len(polygons) == 0:
    raise RuntimeError("No valid element polygons could be built from DFSU geometry.")

# keep only valid elements in values
val_max = val_max[valid_idx]
val_mean = val_mean[valid_idx]
val_min = val_min[valid_idx]

elem_gdf = gpd.GeoDataFrame(
    {
        "elem_id": valid_idx,
        "abs_max": val_max,
        "abs_mean": val_mean,
        "abs_min": val_min
    },
    geometry=polygons,
    crs=f"EPSG:{mesh_epsg}"
)

# spatial index
_ = elem_gdf.sindex

print("Element CRS:", elem_gdf.crs)
print("No. of valid elements:", len(elem_gdf))

# ============================================================
# REPROJECT BAYS TO MESH CRS
# ============================================================
if bays_gdf.crs is None:
    raise ValueError("Bay shapefile has no CRS defined. Please define it before running.")

bays_gdf = bays_gdf.to_crs(elem_gdf.crs)
print("Bay CRS after reprojection:", bays_gdf.crs)

# ============================================================
# COMPUTE AREA-WEIGHTED AVERAGES
# ============================================================
print("\n===== BAY-WISE ABSOLUTE CURRENT SPEED STATISTICS (AREA-WEIGHTED) =====\n")

results = []

for _, row in bays_gdf.iterrows():
    bay_name = row[bay_field]
    bay_geom = row.geometry

    if bay_geom is None or bay_geom.is_empty:
        print(f"{bay_name}: Empty geometry")
        results.append([bay_name, np.nan, np.nan, np.nan])
        continue

    if buffer_tol != 0:
        bay_geom = bay_geom.buffer(buffer_tol)

    # use bounding box first for speed
    minx, miny, maxx, maxy = bay_geom.bounds
    subset = elem_gdf.cx[minx:maxx, miny:maxy].copy()

    # exact geometric filter
    subset = subset[subset.geometry.intersects(bay_geom)].copy()

    print(f"{bay_name}: candidate intersecting elements = {len(subset)}")

    if subset.empty:
        print(f"{bay_name}: No overlapping elements\n")
        results.append([bay_name, np.nan, np.nan, np.nan])
        continue

    # intersection geometry
    inter_geom = subset.geometry.intersection(bay_geom)

    # area of overlap
    area = inter_geom.area.to_numpy(dtype=float)

    # valid overlap mask
    mask = np.isfinite(area) & (area > 0)
    subset = subset.loc[mask].copy()
    area = area[mask]

    print(f"{bay_name}: valid overlap elements = {len(subset)}")
    print(f"{bay_name}: total overlap area = {area.sum():.3f} m²")

    if len(subset) == 0 or area.sum() <= 0:
        print(f"{bay_name}: Zero valid overlap area\n")
        results.append([bay_name, np.nan, np.nan, np.nan])
        continue

    avg_abs_max = area_weighted_mean(subset["abs_max"].values, area)
    avg_abs_mean = area_weighted_mean(subset["abs_mean"].values, area)
    avg_abs_min = area_weighted_mean(subset["abs_min"].values, area)

    results.append([bay_name, avg_abs_max, avg_abs_mean, avg_abs_min])

    print(f"{bay_name}")
    print(f"  Avg |Statistical MAX|  : {avg_abs_max:.3f} m/s")
    print(f"  Avg |Statistical MEAN| : {avg_abs_mean:.3f} m/s")
    print(f"  Avg |Statistical MIN|  : {avg_abs_min:.3f} m/s\n")

# ============================================================
# PRINT FINAL SUMMARY TABLE
# ============================================================
import pandas as pd

res_df = pd.DataFrame(
    results,
    columns=["Bay", "Avg_Abs_Max_mps", "Avg_Abs_Mean_mps", "Avg_Abs_Min_mps"]
)

print("\n===== FINAL SUMMARY =====")
print(res_df.to_string(index=False))


Reading bay polygons...
Bay CRS before reprojection: EPSG:32614

Reading DFSU...

Available DFSU items:
  0: Statistical minimum : Current speed
  1: Statistical maximum : Current speed
  2: Statistical mean : Current speed

NaN counts in DFSU values:
  abs(max) : 7354
  abs(mean): 7354
  abs(min) : 7354

Building DFSU element polygons...
Element CRS: EPSG:32614
No. of valid elements: 203088
Bay CRS after reprojection: EPSG:32614

===== BAY-WISE ABSOLUTE CURRENT SPEED STATISTICS (AREA-WEIGHTED) =====

Aransas: candidate intersecting elements = 28202
Aransas: valid overlap elements = 28202
Aransas: total overlap area = 476960993.446 m²
Aransas
  Avg |Statistical MAX|  : 1.988 m/s
  Avg |Statistical MEAN| : 0.455 m/s
  Avg |Statistical MIN|  : 0.000 m/s

Baffin: candidate intersecting elements = 26614
Baffin: valid overlap elements = 26614
Baffin: total overlap area = 487762758.306 m²
Baffin
  Avg |Statistical MAX|  : 0.950 m/s
  Avg |Statistical MEAN| : 0.348 m/s
  Avg |Statistical MIN

In [18]:
import numpy as np
import pandas as pd
import geopandas as gpd
import mikeio
from shapely.geometry import Polygon
from shapely.validation import make_valid

# ============================================================
# INPUT PATHS
# ============================================================
bay_shp_zip = r"D:\Phd Research\GIS\Shape\Major_Bays_polygon.zip"
dfsu_file = r"F:\wave_result_normal_condition_sig_wv_ht.dfsu"

# ============================================================
# SETTINGS
# ============================================================
bay_field = "bay"
target_bays = ["Matagorda", "San Antonio", "Aransas", "Corpus Christi", "Baffin"]

# Set correct CRS of DFSU mesh
mesh_epsg = 32614   # UTM Zone 14N

# Small optional buffer for robust intersection, in map units (meters)
buffer_tol = 0.0

# ============================================================
# HELPER FUNCTIONS
# ============================================================
def safe_make_polygon(coords):
    """Create valid polygon; return None if impossible."""
    try:
        poly = Polygon(coords)
        if not poly.is_valid:
            poly = make_valid(poly)
        if poly.is_empty:
            return None
        return poly
    except Exception:
        return None


def area_weighted_mean(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not np.any(mask):
        return np.nan
    v = values[mask]
    w = weights[mask]
    wsum = w.sum()
    if wsum <= 0:
        return np.nan
    return np.sum(v * w) / wsum


def find_item_index(item_names, candidates):
    for c in candidates:
        if c in item_names:
            return item_names.index(c)
    return None

# ============================================================
# READ BAY POLYGONS
# ============================================================
print("\nReading bay polygons...")
bays_gdf = gpd.read_file(f"zip://{bay_shp_zip}")

if bay_field not in bays_gdf.columns:
    raise ValueError(f"Field '{bay_field}' not found. Available fields: {list(bays_gdf.columns)}")

bays_gdf = bays_gdf[bays_gdf[bay_field].isin(target_bays)].copy()
if bays_gdf.empty:
    raise ValueError("No target bays found in shapefile.")

bays_gdf = bays_gdf.dissolve(by=bay_field).reset_index()

# Fix invalid geometries
bays_gdf["geometry"] = bays_gdf["geometry"].apply(
    lambda g: make_valid(g) if g is not None and not g.is_valid else g
)
bays_gdf = bays_gdf[bays_gdf.geometry.notnull() & ~bays_gdf.geometry.is_empty].copy()

print("Bay CRS before reprojection:", bays_gdf.crs)

# ============================================================
# READ DFSU
# ============================================================
print("\nReading DFSU...")
dfs = mikeio.open(dfsu_file)
geom = dfs.geometry

item_names = [it.name for it in dfs.items]
print("\nAvailable DFSU items:")
for i, name in enumerate(item_names):
    print(f"  {i}: {name}")

# ------------------------------------------------------------
# Robust item matching for Significant Wave Height
# ------------------------------------------------------------
idx_max = find_item_index(item_names, [
    "Statistical maximum : Significant wave height",
    "Statistical maximum: Significant wave height",
    "Statistical maximum : Sign. Wave Height",
    "Statistical maximum: Sign. Wave Height",
    "Maximum significant wave height"
])

idx_mean = find_item_index(item_names, [
    "Statistical mean : Significant wave height",
    "Statistical mean: Significant wave height",
    "Statistical mean : Sign. Wave Height",
    "Statistical mean: Sign. Wave Height",
    "Mean significant wave height"
])

idx_min = find_item_index(item_names, [
    "Statistical minimum : Significant wave height",
    "Statistical minimum: Significant wave height",
    "Statistical minimum : Sign. Wave Height",
    "Statistical minimum: Sign. Wave Height",
    "Minimum significant wave height"
])

if idx_max is None or idx_mean is None or idx_min is None:
    raise ValueError(
        "Could not find one or more Significant Wave Height items.\n"
        f"Found max={idx_max}, mean={idx_mean}, min={idx_min}"
    )

ds = dfs.read(items=[idx_max, idx_mean, idx_min])

# Statistical dfsu usually has one timestep
val_max = np.asarray(ds[0].values[0], dtype=float)
val_mean = np.asarray(ds[1].values[0], dtype=float)
val_min = np.asarray(ds[2].values[0], dtype=float)

print("\nNaN counts in DFSU values:")
print("  max :", np.isnan(val_max).sum())
print("  mean:", np.isnan(val_mean).sum())
print("  min :", np.isnan(val_min).sum())

# ============================================================
# BUILD ELEMENT POLYGONS
# ============================================================
print("\nBuilding DFSU element polygons...")
node_xy = geom.node_coordinates[:, :2]
elem_table = geom.element_table

polygons = []
valid_idx = []

for i, nodes in enumerate(elem_table):
    coords = node_xy[np.asarray(nodes)]
    poly = safe_make_polygon(coords)
    if poly is not None:
        polygons.append(poly)
        valid_idx.append(i)

if len(polygons) == 0:
    raise RuntimeError("No valid element polygons could be built from DFSU geometry.")

# Filter values to valid polygons only
val_max = val_max[valid_idx]
val_mean = val_mean[valid_idx]
val_min = val_min[valid_idx]

elem_gdf = gpd.GeoDataFrame(
    {
        "elem_id": valid_idx,
        "max_hs": val_max,
        "mean_hs": val_mean,
        "min_hs": val_min
    },
    geometry=polygons,
    crs=f"EPSG:{mesh_epsg}"
)

_ = elem_gdf.sindex

print("Element CRS:", elem_gdf.crs)
print("No. of valid elements:", len(elem_gdf))

# ============================================================
# REPROJECT BAYS TO MESH CRS
# ============================================================
if bays_gdf.crs is None:
    raise ValueError("Bay shapefile has no CRS defined.")

bays_gdf = bays_gdf.to_crs(elem_gdf.crs)
print("Bay CRS after reprojection:", bays_gdf.crs)

# ============================================================
# COMPUTE AREA-WEIGHTED AVERAGES
# ============================================================
print("\n===== BAY-WISE SIGNIFICANT WAVE HEIGHT STATISTICS (AREA-WEIGHTED) =====\n")

results = []

for _, row in bays_gdf.iterrows():
    bay_name = row[bay_field]
    bay_geom = row.geometry

    if bay_geom is None or bay_geom.is_empty:
        print(f"{bay_name}: Empty geometry")
        results.append([bay_name, np.nan, np.nan, np.nan])
        continue

    if buffer_tol != 0:
        bay_geom = bay_geom.buffer(buffer_tol)

    # Bounding box filter first
    minx, miny, maxx, maxy = bay_geom.bounds
    subset = elem_gdf.cx[minx:maxx, miny:maxy].copy()

    # Exact intersection
    subset = subset[subset.geometry.intersects(bay_geom)].copy()

    print(f"{bay_name}: candidate intersecting elements = {len(subset)}")

    if subset.empty:
        print(f"{bay_name}: No overlapping elements\n")
        results.append([bay_name, np.nan, np.nan, np.nan])
        continue

    inter_geom = subset.geometry.intersection(bay_geom)
    area = inter_geom.area.to_numpy(dtype=float)

    mask = np.isfinite(area) & (area > 0)
    subset = subset.loc[mask].copy()
    area = area[mask]

    print(f"{bay_name}: valid overlap elements = {len(subset)}")
    print(f"{bay_name}: total overlap area = {area.sum():.3f} m²")

    if len(subset) == 0 or area.sum() <= 0:
        print(f"{bay_name}: Zero valid overlap area\n")
        results.append([bay_name, np.nan, np.nan, np.nan])
        continue

    avg_max = area_weighted_mean(subset["max_hs"].values, area)
    avg_mean = area_weighted_mean(subset["mean_hs"].values, area)
    avg_min = area_weighted_mean(subset["min_hs"].values, area)

    results.append([bay_name, avg_max, avg_mean, avg_min])

    print(f"{bay_name}")
    print(f"  Avg Statistical MAX  : {avg_max:.3f} m")
    print(f"  Avg Statistical MEAN : {avg_mean:.3f} m")
    print(f"  Avg Statistical MIN  : {avg_min:.3f} m\n")

# ============================================================
# FINAL SUMMARY TABLE
# ============================================================
res_df = pd.DataFrame(
    results,
    columns=["Bay", "Avg_Max_Hs_m", "Avg_Mean_Hs_m", "Avg_Min_Hs_m"]
)

print("\n===== FINAL SUMMARY =====")
print(res_df.to_string(index=False))


Reading bay polygons...
Bay CRS before reprojection: EPSG:32614

Reading DFSU...

Available DFSU items:
  0: Statistical minimum : Sign. Wave Height
  1: Statistical maximum : Sign. Wave Height
  2: Statistical mean : Sign. Wave Height

NaN counts in DFSU values:
  max : 11803
  mean: 11803
  min : 11803

Building DFSU element polygons...
Element CRS: EPSG:32614
No. of valid elements: 203088
Bay CRS after reprojection: EPSG:32614

===== BAY-WISE SIGNIFICANT WAVE HEIGHT STATISTICS (AREA-WEIGHTED) =====

Aransas: candidate intersecting elements = 28202
Aransas: valid overlap elements = 28202
Aransas: total overlap area = 476960993.446 m²
Aransas
  Avg Statistical MAX  : 0.287 m
  Avg Statistical MEAN : 0.011 m
  Avg Statistical MIN  : 0.004 m

Baffin: candidate intersecting elements = 26614
Baffin: valid overlap elements = 26614
Baffin: total overlap area = 487762758.306 m²
Baffin
  Avg Statistical MAX  : 0.269 m
  Avg Statistical MEAN : 0.009 m
  Avg Statistical MIN  : 0.004 m

Corpus 